# Carga de datos

Este notebook lee `Base_de_datos.xlsx` y deja listo un DataFrame `df` con los tipos de dato correctos.

**Cómo usarlo**

1. Instala las dependencias (`pip install -r requirements.txt`). Para leer `.xlsx` hace falta `openpyxl`.
2. Deja `Base_de_datos.xlsx` en la raíz del proyecto, o en cualquier carpeta por encima de este notebook. La función `buscar_archivo` lo encuentra sola.
3. Ejecuta todas las celdas. Para reutilizar los datos desde otro notebook (por ejemplo `comprension_eda.ipynb`):

```python
%run ./Cargar_datos.ipynb
# ahora `df` y la función `cargar_datos()` están disponibles
```

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Ubicar el archivo

Se busca `Base_de_datos.xlsx` subiendo desde la carpeta actual. Así el notebook funciona sin importar desde dónde se abra Jupyter.

In [2]:
NOMBRE_ARCHIVO = "Base_de_datos.xlsx"


def buscar_archivo(nombre: str = NOMBRE_ARCHIVO, desde: Path | None = None) -> Path:
    """Busca `nombre` en la carpeta actual y en todas sus carpetas padre."""
    desde = Path(desde or Path.cwd()).resolve()
    for carpeta in [desde, *desde.parents]:
        candidato = carpeta / nombre
        if candidato.exists():
            return candidato
    raise FileNotFoundError(f"No se encontró '{nombre}' desde {desde} hacia arriba.")


RUTA_DATOS = buscar_archivo()
RUTA_DATOS

WindowsPath('C:/Users/ammie/Documents/1. HENRY/Modulo_5/Proyecto Integrador M5/Base_de_datos.xlsx')

## 2. Diccionario de columnas

| Columna | Descripción | Tipo |
|---|---|---|
| `tipo_credito` | Código del tipo de crédito | categórica (código numérico) |
| `fecha_prestamo` | Fecha de desembolso | fecha |
| `capital_prestado` | Monto prestado | numérica |
| `plazo_meses` | Plazo del crédito en meses | numérica |
| `edad_cliente` | Edad del cliente | numérica |
| `tipo_laboral` | Empleado / Independiente | categórica |
| `salario_cliente` | Salario declarado | numérica |
| `total_otros_prestamos` | Monto de otros préstamos | numérica |
| `cuota_pactada` | Cuota mensual pactada | numérica |
| `puntaje` | Puntaje interno | numérica |
| `puntaje_datacredito` | Puntaje de la central de riesgo | numérica |
| `cant_creditosvigentes` | Número de créditos vigentes | numérica |
| `huella_consulta` | Consultas recientes en la central | numérica |
| `saldo_mora` | Saldo en mora | numérica |
| `saldo_total` | Saldo total de deudas | numérica |
| `saldo_principal` | Saldo de capital | numérica |
| `saldo_mora_codeudor` | Saldo en mora como codeudor | numérica |
| `creditos_sectorFinanciero` | Créditos en el sector financiero | numérica |
| `creditos_sectorCooperativo` | Créditos en el sector cooperativo | numérica |
| `creditos_sectorReal` | Créditos en el sector real | numérica |
| `promedio_ingresos_datacredito` | Ingreso promedio según la central | numérica |
| `tendencia_ingresos` | Creciente / Estable / Decreciente | categórica |
| `Pago_atiempo` | **Variable objetivo**: 1 = pagó a tiempo, 0 = no | binaria |

In [3]:
COLUMNAS_CATEGORICAS = ["tipo_credito", "tipo_laboral", "tendencia_ingresos"]
COLUMNA_FECHA = "fecha_prestamo"
TARGET = "Pago_atiempo"

## 3. Función de carga

`cargar_datos()` lee el Excel y ajusta los tipos de dato:
- `fecha_prestamo` → `datetime`. Si Excel la guardó como número de serie, se convierte desde el origen de Excel (1899-12-30).
- Columnas categóricas → `category`.
- `Pago_atiempo` → entero.

In [4]:
def cargar_datos(ruta: Path | str | None = None, hoja: str | int = 0) -> pd.DataFrame:
    """Lee la base de datos de créditos y devuelve un DataFrame con tipos corregidos."""
    ruta = Path(ruta) if ruta else buscar_archivo()
    datos = pd.read_excel(ruta, sheet_name=hoja, engine="openpyxl")

    if pd.api.types.is_numeric_dtype(datos[COLUMNA_FECHA]):
        datos[COLUMNA_FECHA] = pd.to_datetime(datos[COLUMNA_FECHA], unit="D", origin="1899-12-30")
    else:
        datos[COLUMNA_FECHA] = pd.to_datetime(datos[COLUMNA_FECHA], errors="coerce")

    for col in COLUMNAS_CATEGORICAS:
        datos[col] = datos[col].astype("category")

    datos[TARGET] = datos[TARGET].astype(int)
    return datos


df = cargar_datos(RUTA_DATOS)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

Filas: 10,763 | Columnas: 23


## 4. Verificación rápida

In [5]:
df.head()

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,"3,692,160.00",10,42,Independiente,8000000,2500000,341296,88.77,695.00,10,5,0.00,"51,258.00","51,258.00",0.00,5,0,0,"908,526.00",Estable,1
1,4,2025-04-22 09:47:35,"840,000.00",6,60,Empleado,3000000,2000000,124876,95.23,789.00,3,1,0.00,"8,673.00","8,673.00",0.00,0,0,2,"939,017.00",Creciente,1
2,9,2026-01-08 12:22:40,"5,974,028.40",10,36,Independiente,4036000,829000,529554,47.61,740.00,4,5,0.00,"18,702.00","18,702.00",0.00,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,"1,671,240.00",6,48,Empleado,1524547,498000,252420,95.23,837.00,4,4,0.00,"15,782.00","15,782.00",0.00,3,0,0,"1,536,193.00",Creciente,1
4,9,2025-04-26 11:24:26,"2,781,636.00",11,44,Empleado,5000000,4000000,217037,95.23,771.00,4,6,0.00,"204,804.00","204,804.00",0.00,3,0,1,"933,473.00",Creciente,1


In [6]:
df.dtypes.to_frame("tipo")

,tipo
tipo_credito,category
fecha_prestamo,datetime64[ns]
capital_prestado,float64
plazo_meses,int64
edad_cliente,int64
tipo_laboral,category
salario_cliente,int64
total_otros_prestamos,int64
cuota_pactada,int64
puntaje,float64


In [7]:
df.isna().sum().loc[lambda s: s > 0].to_frame("nulos")

,nulos
puntaje_datacredito,6
saldo_mora,156
saldo_total,156
saldo_principal,405
saldo_mora_codeudor,590
promedio_ingresos_datacredito,2930
tendencia_ingresos,2932
